# Insect Recognation System

Single-Class Insect Detection — Training Guide (COCO / optional YOLO→COCO)

This guide trains a Torchvision Faster R-CNN model for single-class detection (class: insect).
You can start from either:

- COCO format (recommended): images/ + annotations.json

- YOLO format: images/ + labels/ (.txt with normalized bbox), then convert to COCO (optional section)

After setting paths and flags in the config cell, you should be able to run all code cells in order.

# Environment Setup

## Requirements

- Python 3.9+

- If you have a GPU + CUDA, training will be much faster.

Install dependencies (choose one approach):

- Use your environment manager (conda/venv), then install pip packages.

In [ ]:
# If you already have these installed, you can skip this cell.

!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U pillow tqdm torchmetrics


# Configuration (EDIT ONLY THIS CELL)

In this cell you choose:

- whether to convert YOLO→COCO

- where your dataset lives

- where outputs (converted COCO + model checkpoints) should be saved

Default is single-class (“insect”, id=1).
To train multi-class, your COCO annotations.json must contain multiple categories entries (ids starting at 1), and each annotation must use the correct category_id. If you convert from YOLO, edit the converter section where categories and class_id_map are defined.

In [ ]:
from pathlib import Path

# =========================
# A) Choose input format
# =========================
# If your customer already has COCO, keep this False.
RUN_YOLO_TO_COCO = False

# =========================
# B) Paths (edit these)
# =========================

# (1) If RUN_YOLO_TO_COCO = True:
# Expecting:
#   YOLO_DATASET_ROOT/
#     train/images/*.jpg + train/labels/*.txt
#     val/images/*.jpg   + val/labels/*.txt   (optional)
#     test/images/*.jpg  + test/labels/*.txt  (optional)
YOLO_DATASET_ROOT = Path("../dataset/dataset_splits")  # <- change as needed

# Where the converted COCO dataset will be written
COCO_OUTPUT_ROOT = Path("../dataset/splitted_coco_dataset_single_class")  # <- change as needed

# (2) If RUN_YOLO_TO_COCO = False:
# Expecting COCO_ROOT like:
#   COCO_ROOT/
#     train/images/*.jpg
#     train/annotations.json
#     val/images/*.jpg       (optional)
#     val/annotations.json   (optional)
#     test/images/*.jpg      (optional)
#     test/annotations.json  (optional)
COCO_ROOT = Path("../dataset/splitted_coco_dataset_single_class")  # <- change as needed

# =========================
# C) Training hyperparameters
# =========================
BATCH_SIZE = 2
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # set >0 if your environment supports it

# =========================
# D) Output / checkpoints
# =========================
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
LAST_MODEL_PATH = OUTPUT_DIR / "last_model.pt"


# (Optional) YOLO → COCO Conversion

Only run this section if:

- RUN_YOLO_TO_COCO = True

What it does:

- Reads YOLO labels (class xc yc w h normalized)

- Converts to COCO bbox (x, y, width, height in pixels)

- Copies images into COCO folder structure

- Creates annotations.json per split

- Remaps YOLO class IDs into COCO category IDs (starting at 1)

Note (single-class default): This converter currently writes COCO categories as only one class: insect (id=1) and maps all YOLO classes into that single class.
If you want multi-class COCO output, you must modify:

- categories = [...] (add all class names + ids)

- class_id_map (preserve YOLO class ids → COCO ids mapping)

- category_id assignment in annotations

In [ ]:
import os
import json
import shutil
from PIL import Image
from collections import defaultdict

SPLITS = ["train", "val", "test"]
IMG_EXTS = [".jpg", ".jpeg", ".png"]

def convert_yolo_split_to_coco(dataset_root: Path, output_root: Path, split: str):
    """
    Converts one split:
      dataset_root/split/images
      dataset_root/split/labels
    into:
      output_root/split/images
      output_root/split/annotations.json
    """
    img_in = dataset_root / split / "images"
    lbl_in = dataset_root / split / "labels"

    if not img_in.exists():
        print(f"[SKIP] Split '{split}' has no images folder:", img_in)
        return False

    img_out = output_root / split / "images"
    img_out.mkdir(parents=True, exist_ok=True)

    out_ann = output_root / split / "annotations.json"

    # -------- collect unique class IDs from YOLO labels --------
    all_class_ids = set()
    if lbl_in.exists():
        for f in os.listdir(lbl_in):
            if not f.endswith(".txt"):
                continue
            with open(lbl_in / f) as fh:
                for line in fh:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    cls_id = int(float(parts[0]))
                    all_class_ids.add(cls_id)

    # map YOLO class ids -> COCO ids (starting at 1)
    sorted_ids = sorted(list(all_class_ids))
    class_id_map = {orig: i + 1 for i, orig in enumerate(sorted_ids)}

    # If dataset might be empty or labels missing, keep single-class fallback:
    # COCO single class must be id=1 named "insect".
    # If no labels found, we still create categories with that default.
    if len(class_id_map) == 0:
        class_id_map = {0: 1}

    categories = [{"id": 1, "name": "insect"}]

    images = []
    annotations = []
    ann_id = 1
    img_id = 1

    # -------- iterate images --------
    for img_name in os.listdir(img_in):
        if not any(img_name.lower().endswith(ext) for ext in IMG_EXTS):
            continue

        src_img = img_in / img_name
        dst_img = img_out / img_name
        shutil.copy2(src_img, dst_img)

        with Image.open(src_img) as img:
            width, height = img.size

        images.append({
            "id": img_id,
            "file_name": img_name,
            "width": width,
            "height": height
        })

        label_path = lbl_in / (Path(img_name).stem + ".txt")

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    orig_cls, xc, yc, w, h = map(float, line.split())
                    orig_cls = int(orig_cls)

                    coco_cls = class_id_map.get(orig_cls, 1)

                    x = (xc - w / 2) * width
                    y = (yc - h / 2) * height
                    bw = w * width
                    bh = h * height

                    annotations.append({
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": coco_cls,  # single-class (insect)
                        "bbox": [x, y, bw, bh],
                        "area": float(bw * bh),
                        "iscrowd": 0
                    })
                    ann_id += 1

        img_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    with open(out_ann, "w", encoding="utf-8") as f:
        json.dump(coco, f, indent=2)

    print(f"[DONE] {split}: Images={len(images)} | Annotations={len(annotations)} | Saved={out_ann}")
    return True


if RUN_YOLO_TO_COCO:
    print("YOLO input :", YOLO_DATASET_ROOT)
    print("COCO output:", COCO_OUTPUT_ROOT)

    COCO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for split in SPLITS:
        convert_yolo_split_to_coco(YOLO_DATASET_ROOT, COCO_OUTPUT_ROOT, split)

    # After conversion, we will train from this COCO folder:
    COCO_ROOT = COCO_OUTPUT_ROOT
    print("\nConversion complete. Training will use COCO_ROOT =", COCO_ROOT)
else:
    print("RUN_YOLO_TO_COCO is False — skipping conversion.")
    print("Training will use COCO_ROOT =", COCO_ROOT)


# Validate COCO Dataset Structure

This section checks that COCO paths exist.
Minimum required:

- train/images/

- train/annotations.json

val/ and test/ are optional.

In [ ]:
from pathlib import Path

TRAIN_JSON = COCO_ROOT / "train" / "annotations.json"
VAL_JSON   = COCO_ROOT / "val"   / "annotations.json"
TEST_JSON  = COCO_ROOT / "test"  / "annotations.json"

TRAIN_IMG_DIR = COCO_ROOT / "train" / "images"
VAL_IMG_DIR   = COCO_ROOT / "val"   / "images"
TEST_IMG_DIR  = COCO_ROOT / "test"  / "images"

print("CWD:", Path.cwd())
print("COCO_ROOT:", COCO_ROOT)

print("Train JSON:", TRAIN_JSON)
print("Train IMG :", TRAIN_IMG_DIR)

assert TRAIN_JSON.exists(), f"Missing: {TRAIN_JSON}"
assert TRAIN_IMG_DIR.exists(), f"Missing: {TRAIN_IMG_DIR}"

print("\nOptional splits:")
print("Val  JSON exists?", VAL_JSON.exists(), "| Val images exists?", VAL_IMG_DIR.exists())
print("Test JSON exists?", TEST_JSON.exists(), "| Test images exists?", TEST_IMG_DIR.exists())


# COCO Dataset Loader (Single-Class by default)

labels are set to 1 (insect).

This dataset loader:

- reads COCO annotations.json

returns:

- image (PIL → tensor via transforms)

- target dictionary containing:

- - boxes in [x1,y1,x2,y2]

- - labels (all 1 for insect)

- - image_id, area, iscrowd

It also supports images without annotations (valid negatives).

#### Important (Multi-class note):
This notebook is written for single-class detection. Even if your COCO file contains multiple categories, the current loader forces all labels to 1.

If you want multi-class training, you must change the loader so that:

- labels are read from COCO annotation["category_id"] instead of always 1.

(See the “Model” section as well: num_classes must also be updated for multi-class.)



In [ ]:
import os, json
from typing import Dict, List

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CocoSingleClassDataset(Dataset):
    def __init__(self, img_dir: Path, ann_path: Path, transforms=None):
        self.img_dir = Path(img_dir)
        self.ann_path = Path(ann_path)
        self.transforms = transforms

        with open(self.ann_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        # Map image_id -> image info
        self.images = coco["images"]
        self.id_to_img = {img["id"]: img for img in self.images}

        # Group annotations by image_id
        self.ann_by_img: Dict[int, List[dict]] = {img["id"]: [] for img in self.images}
        for ann in coco.get("annotations", []):
            self.ann_by_img[ann["image_id"]].append(ann)

        self.image_ids = [img["id"] for img in self.images]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        image_id = self.image_ids[idx]
        img_info = self.id_to_img[image_id]

        img_path = self.img_dir / img_info["file_name"]
        image = Image.open(img_path).convert("RGB")

        anns = self.ann_by_img.get(image_id, [])

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(1)  # single class insect
            areas.append(float(ann.get("area", w * h)))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id]),
            "area": areas,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


# Transforms

Torchvision v2 transforms:

- converts PIL image → tensor

- scales to float32 in [0..1]

In [ ]:
import torchvision
from torchvision.transforms import v2 as T

train_tfms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

val_tfms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])


# DataLoaders

Creates loaders for:

- train (required)

- val (optional)

- test (optional)

In [ ]:
train_ds = CocoSingleClassDataset(TRAIN_IMG_DIR, TRAIN_JSON, transforms=train_tfms)

val_ds = None
if VAL_JSON.exists() and VAL_IMG_DIR.exists():
    val_ds = CocoSingleClassDataset(VAL_IMG_DIR, VAL_JSON, transforms=val_tfms)

test_ds = None
if TEST_JSON.exists() and TEST_IMG_DIR.exists():
    test_ds = CocoSingleClassDataset(TEST_IMG_DIR, TEST_JSON, transforms=val_tfms)

print("Train images:", len(train_ds))
print("Val images  :", len(val_ds) if val_ds else "N/A")
print("Test images :", len(test_ds) if test_ds else "N/A")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

val_loader = None
if val_ds:
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn
    )

test_loader = None
if test_ds:
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn
    )

print("Loaders ready.")


# Model — Faster R-CNN (single class) 

#### Model — Faster R-CNN (Single-Class by default)

We train Faster R-CNN with:

- Backbone: ResNet50-FPN (Torchvision)

- Head: replaced for num_classes = 2 (background + insect)

#### Important (Multi-class note):
This notebook is configured for one class (“insect”).

If you want multi-class detection, then:

- Your COCO annotations.json must include multiple entries in categories

- Each annotation must use the correct category_id

- The model must be created with:

num_classes = 1 + number_of_categories
(background + all classes)

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

num_classes = 2  # background + insect

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(device)
print("Model ready.")


# Training Loop

This training loop:

- trains for EPOCHS

- if validation exists: saves the best checkpoint to BEST_MODEL_PATH

- otherwise: saves last checkpoint to LAST_MODEL_PATH

In [ ]:
import torch.optim as optim
import time

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    running = 0.0
    n = 0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running += float(losses.item())
        n += 1

    return running / max(n, 1)


### Run training (+ optional early stopping)

In [ ]:
EPOCHS = int(EPOCHS)
best_val = float("inf")

patience = 5
min_delta = 0.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
    lr_scheduler.step()

    msg = f"Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f}"

    if val_loader is not None:
        model.eval()
        val_running = 0.0
        val_n = 0

        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                val_running += float(losses.item())
                val_n += 1

        val_loss = val_running / max(val_n, 1)
        msg += f" | val_loss={val_loss:.4f}"

        improved = (best_val - val_loss) > min_delta
        if improved:
            best_val = val_loss
            bad_epochs = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            msg += f" | saved {BEST_MODEL_PATH.name}"
        else:
            bad_epochs += 1
            msg += f" | no_improve={bad_epochs}/{patience}"

        if bad_epochs >= patience:
            msg += " | EARLY STOP"
            msg += f" | time={(time.time() - t0):.1f}s"
            print(msg)
            break

    else:
        torch.save(model.state_dict(), LAST_MODEL_PATH)
        msg += f" | saved {LAST_MODEL_PATH.name}"

    msg += f" | time={(time.time() - t0):.1f}s"
    print(msg)

# Load best model if validation existed
if val_loader is not None and BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    print("Loaded best model:", BEST_MODEL_PATH)
elif val_loader is None and LAST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(LAST_MODEL_PATH, map_location=device))
    print("Loaded last model:", LAST_MODEL_PATH)


# Inference: “Insect present?” (yes/no)

This helper returns:

- present (bool)

- max_score

- raw detection outputs

In [ ]:
from PIL import Image

@torch.no_grad()
def predict_insect_present(model, image_path: str, threshold: float = 0.5):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    x = val_tfms(img).to(device)

    outputs = model([x])[0]
    scores = outputs["scores"].detach().cpu()

    present = bool((scores >= threshold).any().item()) if len(scores) else False
    max_score = float(scores.max().item()) if len(scores) else 0.0

    return present, max_score, outputs


# Quick Inference Test

Set a filename from your dataset and test.

In [ ]:
# Pick an image filename that exists under TEST_IMG_DIR or VAL_IMG_DIR or TRAIN_IMG_DIR
FILENAME = None  # e.g. "example.jpg"

if FILENAME is None:
    print("Set FILENAME to a real image name before running this cell.")
else:
    # Prefer test, then val, then train
    candidate_paths = [
        TEST_IMG_DIR / FILENAME,
        VAL_IMG_DIR / FILENAME,
        TRAIN_IMG_DIR / FILENAME,
    ]
    test_image = next((p for p in candidate_paths if p.exists()), None)
    assert test_image is not None, f"Could not find {FILENAME} in test/val/train images."

    present, max_score, outputs = predict_insect_present(model, str(test_image), threshold=0.5)
    print("Image:", test_image)
    print("Insect present?", present, "| max_score=", round(max_score, 4))


# Evaluation (Optional): mAP with TorchMetrics

If you have a val_loader or test_loader, you can compute detection metrics:

- mAP@[.50:.95]

- mAP@50

- mAP@75

- mar@100

In [ ]:
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

@torch.no_grad()
def evaluate_map_torchmetrics(model, data_loader, device, score_thr=0.05):
    metric = MeanAveragePrecision(iou_type="bbox")
    model.eval()

    for images, targets in tqdm(data_loader, desc="Evaluating mAP"):
        images = [img.to(device) for img in images]
        outputs = model(images)

        preds, gts = [], []
        for out, tgt in zip(outputs, targets):
            boxes  = out["boxes"].detach().cpu()
            scores = out["scores"].detach().cpu()
            labels = out["labels"].detach().cpu()

            if len(scores):
                keep = scores >= score_thr
                boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

            preds.append({"boxes": boxes, "scores": scores, "labels": labels})
            gts.append({
                "boxes": tgt["boxes"].detach().cpu(),
                "labels": tgt["labels"].detach().cpu()
            })

        metric.update(preds, gts)

    r = metric.compute()
    return {
        "mAP_50_95": float(r["map"]),
        "mAP_50": float(r["map_50"]),
        "mAP_75": float(r["map_75"]),
        "mar_100": float(r["mar_100"]),
    }


### Evaluate on val/test (if available)

In [ ]:
if val_loader is not None:
    results_val = evaluate_map_torchmetrics(model, val_loader, device, score_thr=0.05)
    print("VAL mAP:", results_val)
else:
    print("No validation split found — skipping VAL evaluation.")

if test_loader is not None:
    results_test = evaluate_map_torchmetrics(model, test_loader, device, score_thr=0.05)
    print("TEST mAP:", results_test)
else:
    print("No test split found — skipping TEST evaluation.")


# Deliverables / What to hand over

After training:

- outputs/best_model.pt (if val split exists)

- or outputs/last_model.pt (if no val split)

These are state_dict checkpoints that can be loaded into the same model definition.

### How to reload the saved model later

In [ ]:
# Example reload pattern (same model architecture must be created first)
# model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
# model.to(device)
# model.eval()

print("Best model path:", BEST_MODEL_PATH, "| exists:", BEST_MODEL_PATH.exists())
print("Last model path:", LAST_MODEL_PATH, "| exists:", LAST_MODEL_PATH.exists())
